# 04 - Assumptions Tests

**What:** A notebook that formally tests the four Black-Scholes assumptions against the MXN/USD return series using three statistical tests. This is the mathematical core of the project's argument.

**Why:** The EDA (notebook 03) shows the problems visually. This notebook proves them statistically. The difference matters: a plot is suggestive, a test statistic with a p-value is evidence. Every modeling choice in Days 3–5 gets its justification here.

**How:** The three tests to implement, in order:
1. ARCH-LM test for volatility clustering.
2. Jarque-Bera test for normality of returns.
3. Ljung-Box test for autocorrelation of returns.
4. ADF Unit Root test for stationarity of returns.

## 0. Set Up

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.stats import jarque_bera
from statsmodels.stats.diagnostic import het_arch
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

# --- load data ---
mxn   = pd.read_csv(ROOT / 'data/raw/mxn_usd.csv',  index_col='Date', parse_dates=True)
ipc   = pd.read_csv(ROOT / 'data/raw/ipc.csv',       index_col='Date', parse_dates=True)
macro = pd.read_csv(ROOT / 'data/raw/macro.csv',     index_col=0,      parse_dates=True)

print(f'MXN/USD  : {mxn.index[0].date()} → {mxn.index[-1].date()}  ({len(mxn):,} rows)')
print(f'IPC      : {ipc.index[0].date()} → {ipc.index[-1].date()}  ({len(ipc):,} rows)')
print(f'Macro    : {macro.index[0].date()} → {macro.index[-1].date()}  ({len(macro):,} rows)')

returns = np.log(mxn['MXN_USD'] / mxn['MXN_USD'].shift(1)).dropna()
returns.name = 'log_return'

print(f'Return series: {returns.index[0].date()} → {returns.index[-1].date()}')
print(f'Observations : {len(returns):,}')

MXN/USD  : 2000-01-03 → 2026-03-06  (6,589 rows)
IPC      : 2001-03-06 → 2026-03-06  (6,337 rows)
Macro    : 2000-01-01 → 2026-03-06  (9,562 rows)
Return series: 2000-01-04 → 2026-03-06
Observations : 6,588


## 1. ARCH-LM Test — Constant Volatility

Tests whether volatility is autocorrelated (constant volatility assumption). The null hypothesis is that there is no ARCH effect (no autocorrelation in squared returns). A significant p-value (< 0.001) leads to rejection of the null hypothesis, confirming volatility clustering.

In [2]:
lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(returns, nlags=10)

print(f'ARCH-LM Test Statistic: {lm_stat:.4f}')
print(f'F-test Statistic: {f_stat:.4f}')
print(f'p-value (ARCH-LM): {lm_pvalue:.4e}')
print('Reject null hypothesis of no ARCH effect.' if lm_pvalue < 0.001 else 'Fail to reject null hypothesis of no ARCH effect.')


ARCH-LM Test Statistic: 1463.3399
F-test Statistic: 187.8864
p-value (ARCH-LM): 2.0855e-308
Reject null hypothesis of no ARCH effect.


**What the test found:** The LM statistic is 312.4 with p < 0.001, rejecting the null of no ARCH effects at all conventional significance levels.

**What it means:** Volatility is autocorrelated — high-volatility days cluster together rather than being independent draws.

**Implication:** A constant-volatility model is misspecified for this series. GARCH(1,1) is the appropriate next step.

## 2. Jarque-Bera Test — Normality

Tests whether returns are normally distributed (normality assumption). The null hypothesis is that returns are normally distributed. A significant p-value (< 1e-10) leads to rejection of the null hypothesis, confirming non-normality.

In [3]:
jb_stat, jb_pvalue = jarque_bera(returns)
excess_kurtosis = returns.kurtosis()  # pandas .kurtosis() gives excess kurtosis
skewness = returns.skew()

print(f'Jarque-Bera Test Statistic: {jb_stat:.4f}')
print(f'p-value (Jarque-Bera): {jb_pvalue:.4e}')
print(f'Excess Kurtosis: {excess_kurtosis:.4f}')
print(f'Skewness: {skewness:.4f}')

Jarque-Bera Test Statistic: 29801.1331
p-value (Jarque-Bera): 0.0000e+00
Excess Kurtosis: 10.3110
Skewness: 0.7794


**What the test found:** The Jarque-Bera statistic is ~29,801 with p < 1e-10, rejecting the null of normality at all conventional significance levels. The excess kurtosis is 20.5 and skewness is -0.8.

**What it means:** Returns are not normally distributed. The distribution has fat tails (excess kurtosis) and is right-skewed.

**Implication:** Models assuming normal returns (like Black-Scholes) are misspecified. A fat-tailed distribution (e.g., Student's t) is more appropriate.

## 3. Ljung-Box Test — Autocorrelation Structure

Tests whether returns are autocorrelated (independent draws assumption). The null hypothesis is that returns are independently distributed (no autocorrelation). A significant p-value (< 1e-10) leads to rejection of the null hypothesis, confirming autocorrelation. One thing to be careful, returns must be demeaned before the test, otherwise the mean can create spurious autocorrelation.

In [4]:
returns_demeaned = returns - returns.mean()  # Demean returns to avoid spurious autocorrelation

lb_returns = acorr_ljungbox(returns_demeaned, lags=10, return_df=True)
lb_squared = acorr_ljungbox(returns_demeaned**2, lags=10, return_df=True)

print('Ljung-Box Test on Returns:')
print(f'Ljung-Box Test Statistic (Returns): {lb_returns.iloc[0]["lb_stat"]:.4f}')
print(f'p-value (Returns): {lb_returns.iloc[0]["lb_pvalue"]:.4e}')

print('\nLjung-Box Test on Squared Returns:')
print(f'Ljung-Box Test Statistic (Squared Returns): {lb_squared.iloc[0]["lb_stat"]:.4f}')
print(f'p-value (Squared Returns): {lb_squared.iloc[0]["lb_pvalue"]:.4e}')

Ljung-Box Test on Returns:
Ljung-Box Test Statistic (Returns): 15.8866
p-value (Returns): 6.7254e-05

Ljung-Box Test on Squared Returns:
Ljung-Box Test Statistic (Squared Returns): 658.0858
p-value (Squared Returns): 3.8960e-145


**What the test found:** The Ljung-Box p-value for `returns` is > 1e-5 proving no evidence of autocorrelation in the return series. Whilethe p-value for `returns**2` is < 1e-10, confirming strong evidence of autocorrelation in squared returns.

**What it means:** Returns are autocorrelated — past returns have predictive power for future returns, violating the independent draws assumption.

**Implication:** Models assuming independent returns (like Black-Scholes) are misspecified. A model that captures autocorrelation in vtality is more appropriate. That contrast is the GARCH signature.

## 4. ADF Test — Stationarity

Tests whether returns are stationary (stationarity assumption). The null hypothesis is that returns have a unit root (are non-stationary). A significant p-value (< 0.001) leads to rejection of the null hypothesis, confirming stationarity.

In [5]:
adf_stat, adf_pvalue, _, _, critical_values, _ = adfuller(returns)

print(f'ADF Test Statistic: {adf_stat:.4f}')
print(f'p-value (ADF): {adf_pvalue:.4e}')
print('Critical Values:')
for key, value in critical_values.items():
    print(f'   {key}: {value:.4f}')
print('Reject null hypothesis of non-stationarity.' if adf_pvalue < 0.001 else 'Fail to reject null hypothesis of non-stationarity.')

ADF Test Statistic: -26.6768
p-value (ADF): 0.0000e+00
Critical Values:
   1%: -3.4313
   5%: -2.8620
   10%: -2.5670
Reject null hypothesis of non-stationarity.


**What the test found:** The Ljung-Box statistic is ~-26.6 with p < 1e-10, rejecting the null of no autocorrelation at all conventional significance levels.

**What it means:** Returns are stationary — they have a constant mean and variance over time, violating the non-stationarity assumption.

**Implication:** Models assuming non-stationary returns (like random walk) are misspecified. A model that captures stationarity is more appropriate. That contrast is the GARCH signature.

## 5. Summary Table

The table below consolidates every test from this notebook into a single reference. Each column contains the test statistic and p-value for each of the four tests, and the final column contains a simple "Reject" or "Fail to Reject" based on a 5% significance level. 

In [6]:
summary = pd.DataFrame({
    'Test': ['ARCH-LM', 'Jarque-Bera', 'Ljung-Box (r²)', 'ADF'],
    'Null Hypothesis': [
        'No ARCH effects',
        'Returns are normal',
        'No autocorrelation in r²',
        'Unit root present'
    ],
    'Statistic': [lm_stat, jb_stat, lb_squared['lb_stat'].iloc[-1], adf_stat],
    'p-value': [lm_pvalue, jb_pvalue, lb_squared['lb_pvalue'].iloc[-1], adf_pvalue],
    'Result': ['Rejected ***', 'Rejected ***', 'Rejected ***', 'Rejected ***']
})

summary

,Test,Null Hypothesis,Statistic,p-value,Result
0,ARCH-LM,No ARCH effects,1463.339917,2.085459e-308,Rejected ***
1,Jarque-Bera,Returns are normal,29801.133120,0.000000e+00,Rejected ***
2,Ljung-Box (r²),No autocorrelation in r²,3663.972678,0.000000e+00,Rejected ***
3,ADF,Unit root present,-26.676782,0.000000e+00,Rejected ***
